# Individual task

## Set up

In [1]:
! pip install rsa
! pip install PIL


[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement PIL (from versions: none)

[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for PIL


In [2]:
# Used imports
import rsa
from PIL import Image

# Signature creation & verification

In [8]:
# Extracting keys
with open("private_key.pem", "rb") as priv_file:
    private_key = rsa.PrivateKey.load_pkcs1(priv_file.read())

with open("public_key.pem", "rb") as pub_file:
    public_key = rsa.PublicKey.load_pkcs1(pub_file.read())

In [9]:
def sign_image(img_path, private_key, output_path):
    """
    Creates and hides signature inside the image based on LSB method
    """
    with open(img_path, "rb") as f:
        image_data = f.read()

    signature = rsa.sign(image_data, private_key, 'SHA-256') 

    img = Image.open(img_path).convert("RGB")
    pixels = list(img.getdata())

    bits = ''.join(f'{byte:08b}' for byte in signature)
    new_pixels = []
    bit_index = 0

    for pixel in pixels:
        r, g, b = pixel
        if bit_index < len(bits):
            r = (r & ~1) | int(bits[bit_index])
            bit_index += 1
        if bit_index < len(bits):
            g = (g & ~1) | int(bits[bit_index])
            bit_index += 1
        if bit_index < len(bits):
            b = (b & ~1) | int(bits[bit_index])
            bit_index += 1
        new_pixels.append((r, g, b))
        if bit_index >= len(bits):
            new_pixels.extend(pixels[len(new_pixels):])  # add all other pixels without changes if any was left after the message was hidden
            break

    img.putdata(new_pixels)
    img.save(output_path)
    
    print(f"Sucsessfully signed!")


In [10]:
sign_image("bird_original.png", private_key, "signed.png")

Sucsessfully signed!


In [11]:
def verify_signature(original_img, signed_img, public_key, sig_len=512):
    """
    Extracts the signature from the signed image,
    and takes original data to verify the signature
    """
    with open(original_img, "rb") as f:
        image_data = f.read()
        
    img = Image.open(signed_img).convert("RGB")
    pixels = list(img.getdata())

    bits = ""
    for pixel in pixels:
        for color in pixel:
            bits += str(color & 1)
            if len(bits) >= sig_len * 8:
                break
        if len(bits) >= sig_len * 8:
            break

    signature_bytes = bytes(int(bits[i:i+8], 2) for i in range(0, len(bits), 8))
    try:
        rsa.verify(image_data, signature_bytes, public_key)
        print("Signature is verified!")
    except rsa.VerificationError:
        print("Signature virification failed")    



In [12]:
verify_signature("bird_original.png", "signed.png", public_key)

Signature is verified!
